[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter5/vectara-query.ipynb)

# Chapter 5: Query and Hallucination Mitigation with Vectara

This notebook demonstrates Vectara's query API with built-in RAG generation, factual consistency scoring, and hallucination correction — key features for building trustworthy RAG applications.

**What you'll learn:**
- Execute RAG queries with Vectara's hybrid search and reranking
- Use factual consistency scores to measure response trustworthiness
- Apply Vectara's hallucination correction API to fix unfaithful responses

**Prerequisites:** Set `VECTARA_API_KEY` as an environment variable. Run `vectara-ingest.ipynb` first to populate the corpus.

## Simple Query

We issue a RAG query against the corpus using Vectara's hybrid search with reranking, and inspect both the generated summary and its factual consistency score.

In [1]:
import requests
import json
import os

Let's define the Vectara corpus key and API key

In [2]:
CORPUS_KEY = "RAGBOOK"
VECTARA_API_KEY = os.getenv("VECTARA_API_KEY")

In [3]:
url = f"https://api.vectara.io/v2/corpora/{CORPUS_KEY}/query"
query_str = "Are pets allowed in the office?"

payload = json.dumps({
  "query": query_str,
  "search": {
    "lexical_interpolation": 0.025,
    "offset": 0,
    "limit": 50,
    "context_configuration": {
      "sentences_before": 2,
      "sentences_after": 2
    },
    "reranker": {
      "type": "customer_reranker",
      "reranker_name": "Rerank_Multilingual_v1"
    }
  },
  "generation": {
    "max_used_search_results": 7,
    "response_language": "eng",
    "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
    "enable_factual_consistency_score": True
  }
})
headers = {
  'Content-Type': 'application/json',
  'Accept': 'application/json',
  'x-api-key': VECTARA_API_KEY
}

response = requests.post(url, headers=headers, data=payload)
res = response.json()
search_results = res['search_results']

print(res['summary'])
print(f"Factual Consistency Score: {res['factual_consistency_score']}")

Pets are allowed in the office at Vectara, but with specific guidelines. Birds are permitted and even encouraged in the workspace, although there are particular rules to follow [2]. However, common household pets such as cats and dogs are not allowed on Vectara campuses [7].
Factual Consistency Score: 0.85546875


> **Note:** The Factual Consistency Score (0–1) measures how well the generated summary is supported by the retrieved documents. A score above 0.85 indicates high confidence that the response is grounded in the source material. Scores below 0.5 suggest the response may contain hallucinated claims worth reviewing.

> **Note:** `lexical_interpolation` controls the balance between neural (semantic) and lexical (keyword) matching in Vectara's hybrid search. A value of 0.025 means the search relies almost entirely on semantic similarity, with only a small lexical boost to catch exact keyword matches. Increase this value if your queries contain domain-specific terms or acronyms that benefit from exact matching.

In [4]:
search_results[:2]

[{'text': "Strict guidelines, ethical practices, and responsible stewardship are non-negotiable. The Community Outreach: We regularly host educational programs, wildlife awareness\ncampaigns, and community engagement activities. Our exotic pets are not just oﬃce mascots; they are ambassadors of our vision and values. Our exotic pet policy is more than a quirky perk; it's a manifestation of our organizational ethos. It adds a touch of adventure, wisdom, and wildness to our daily grind.",
  'score': 0.9776712656021118,
  'part_metadata': {'page': 3,
   'lang': 'eng',
   'section': 30,
   'offset': 0,
   'len': 89},
  'document_metadata': {'Producer': 'Skia/PDF m118 Google Docs Renderer',
   'Title': 'Employee Handbook - Company Pet Policy',
   'title': 'Employee Handbook - Company Pet Policy'},
  'document_id': 'pet_policy'},
 {'text': 'At Vectara, we believe in soaring to new heights, and what better way to embody that spirit than\nby allowing birds into our workspace? Yes, you read tha

## Hallucination correction

Let's take the previous summary (which has a relatively high (good) hallucination rate of 0.76) and in simulate a scenario where it is in fact a hallucination. Do do so we'll change a few things. Can u see the minor changes?

In [5]:
hallucinated_response = """
Pets are allowed in the office at Vectara, but with specific guidelines. 
Birds are permitted and even encouraged in the workspace, but there are no rules to follow [2]. 
However, common household pets like cats and snakes are not allowed on the Vectara campuses [7].
"""

In [6]:
url = f"https://api.vectara.io/v2/hallucination_correctors/correct_hallucinations"
payload = {
  "generated_text": hallucinated_response,
  "query": query_str,
  "documents": [
      {"text": r["text"]}
      for r in search_results
  ],
  "model_name": "vhc-large-1.0"
}
headers = {
  'Content-Type': 'application/json',
  'Accept': 'application/json',
  'x-api-key': VECTARA_API_KEY
}

response = requests.post(url, headers=headers, json=payload)
res = response.json()

print("Original (hallucinated):")
print(hallucinated_response)
print("\nCorrected:")
print(res['corrected_text'])

Original (hallucinated):

Pets are allowed in the office at Vectara, but with specific guidelines. 
Birds are permitted and even encouraged in the workspace, but there are no rules to follow [2]. 
However, common household pets like cats and snakes are not allowed on the Vectara campuses [7].


Corrected:
Pets are allowed in the office at Vectara, but with specific guidelines. 
Birds are permitted and even encouraged in the workspace, but there are specific guidelines to follow [2]. 
However, common household pets like cats and dogs are not allowed on the Vectara campuses [7].


In [7]:
res['corrections']

[{'original_text': 'Birds are permitted and even encouraged in the workspace, but there are no rules to follow [2].',
  'corrected_text': 'Birds are permitted and even encouraged in the workspace, but there are specific guidelines to follow [2].',
  'explanation': 'The source states that there are specific guidelines and peculiarities for the bird policy, contradicting the claim that "there are no rules to follow."'},
 {'original_text': 'However, common household pets like cats and snakes are not allowed on the Vectara campuses [7].',
  'corrected_text': 'However, common household pets like cats and dogs are not allowed on the Vectara campuses [7].',
  'explanation': 'The source explicitly mentions only cats and dogs as forbidden common household pets and does not mention snakes as common household pets. Including snakes as forbidden common household pets is not supported.'}]